# Preview Image

Assets:
  - Lampung            : projects/data-skripsi-473712/assets/Lampung
  - Kalimantan-selatan : projects/data-skripsi-473712/assets/Kalimantan-selatan
  - Petengoran         : projects/data-skripsi-473712/assets/Petengoran


**Import Libraries**

In [1]:
import ee
import geemap
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"   geemap version : {geemap.__version__}")
print(f"   ee version     : {ee.__version__}")

✅ Libraries imported successfully!
   geemap version : 0.36.6
   ee version     : 1.7.4


**GEE Authentication & Initialization**

In [2]:
PROJECT_ID = 'data-skripsi-473712'

try:
    ee.Initialize(project=PROJECT_ID)
    # Quick test
    test = ee.Number(1).getInfo()
    print(f"✅ Connected to Earth Engine")
    print(f"   Project: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️  Not authenticated, starting auth flow...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print(f"✅ Authentication & initialization complete")
    print(f"   Project: {PROJECT_ID}")

✅ Connected to Earth Engine
   Project: data-skripsi-473712


**Configuration**

In [3]:
# REGION_NAME = 'lampung'
# ASSET_ID    = 'projects/data-skripsi-473712/assets/Lampung'


# REGION_NAME = 'kalsel'
# ASSET_ID    = 'projects/data-skripsi-473712/assets/Kalimantan-selatan'


#REGION_NAME = 'petengoran'
#ASSET_ID    = 'projects/data-skripsi-473712/assets/Petengoran'

REGION_NAME = 'Sawit-PDL'
ASSET_ID = 'projects/data-skripsi-473712/assets/sawit-pdl'

# ---- Year ----
YEAR_S2  = 2023
YEAR_AGB = 2023

# ---- Sentinel-2 parameters ----
CLOUD_THRESHOLD = 70        # Max cloud pixel percentage (%)
SCALE_S2        = 10        # Export resolution in meters
SCALE_AGB       = 10        # AGB reproject resolution in meters

# ---- Sentinel-2 bands ----
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

BAND_DESCRIPTIONS = {
    'B2'  : 'Blue (490 nm)',
    'B3'  : 'Green (560 nm)',
    'B4'  : 'Red (665 nm)',
    'B5'  : 'Red Edge 1 (705 nm)',
    'B6'  : 'Red Edge 2 (740 nm)',
    'B7'  : 'Red Edge 3 (783 nm)',
    'B8'  : 'NIR (842 nm)',
    'B8A' : 'NIR Narrow (865 nm)',
    'B11' : 'SWIR 1 (1610 nm)',
    'B12' : 'SWIR 2 (2190 nm)'
}


print(f"  Region       : {REGION_NAME}")
print(f"  Asset ID     : {ASSET_ID}")
print(f"  Year S2      : {YEAR_S2}")
print(f"  Year AGB     : {YEAR_AGB}")
print(f"  Cloud thresh : {CLOUD_THRESHOLD}%")
print(f"  Scale S2     : {SCALE_S2} m")
print(f"  Scale AGB    : {SCALE_AGB} m")
print(f"  Bands        : {S2_BANDS}")

  Region       : Sawit-PDL
  Asset ID     : projects/data-skripsi-473712/assets/sawit-pdl
  Year S2      : 2023
  Year AGB     : 2023
  Cloud thresh : 70%
  Scale S2     : 10 m
  Scale AGB    : 10 m
  Bands        : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']


**Load Region of Interest (ROI)**

In [4]:
print(f"Loading ROI: {ASSET_ID}")

try:
    roi_fc  = ee.FeatureCollection(ASSET_ID)
    roi     = roi_fc.geometry()

    # Get ROI info
    bounds  = roi.bounds().getInfo()['coordinates'][0]
    area_m2 = roi.area().getInfo()
    area_km2 = area_m2 / 1e6

    print(f"\n✅ ROI loaded successfully: {REGION_NAME}")
    print(f"   Approximate area : {area_km2:,.1f} km²")
    print(f"   Bounding box     : {bounds[0]} → {bounds[2]}")
    
    if area_km2 > 50_000:
        print(f"\n⚠️  WARNING: Large region ({area_km2:,.0f} km²)")
        print(f"   Recommendation: Use EXPORT_METHOD = 'drive' for reliable download")
        print(f"   Change config in Cell 4 if needed.")

except Exception as e:
    print(f"❌ Failed to load ROI: {e}")
    raise

Loading ROI: projects/data-skripsi-473712/assets/sawit-pdl

✅ ROI loaded successfully: Sawit-PDL
   Approximate area : 42.0 km²
   Bounding box     : [115.282307776786, -2.491772422230114] → [115.39567601021554, -2.433777357477759]


**Define Helper Functions**

In [5]:
def mask_s2_scl(image):
    """
    Mask Sentinel-2 clouds using Scene Classification Layer (SCL).
    
    SCL classes kept (valid pixels):
      4  = Vegetation
      5  = Bare soils
      6  = Water
      7  = Unclassified
    
    SCL classes removed (invalid pixels):
      3  = Cloud shadow
      8  = Cloud medium probability
      9  = Cloud high probability
      10 = Cirrus
      11 = Snow / Ice
    """
    scl = image.select('SCL')
    valid_mask = (
        scl.eq(4).Or(scl.eq(5))
           .Or(scl.eq(6)).Or(scl.eq(7))
    )
    return image.updateMask(valid_mask).select(S2_BANDS)


def get_monthly_composite(collection, year, month):
    """
    Create monthly median composite from filtered collection.
    
    Args:
        collection : Filtered ee.ImageCollection
        year       : Target year (int)
        month      : Target month 1-12 (int)
        
    Returns:
        ee.Image: Monthly median composite clipped to ROI
    """
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    
    monthly = collection.filterDate(start, end)
    composite = monthly.median().clip(roi)
    
    return composite, monthly.size()

**Build Sentinel-2 Collection and ESA CCI AGB**

In [6]:
print("\n📡 SENTINEL-2 SR HARMONIZED")
print("-"*60)
print(f"  Collection : COPERNICUS/S2_SR_HARMONIZED")
print(f"  Year       : {YEAR_S2}")
print(f"  Cloud max  : {CLOUD_THRESHOLD}%")
print(f"  Region     : {REGION_NAME}")

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(f'{YEAR_S2}-01-01', f'{YEAR_S2}-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
    .map(mask_s2_scl)
)

# Count total scenes
total_scenes = s2_collection.size().getInfo()
print(f"\n  ✅ Collection built")
print(f"  Total scenes after filter: {total_scenes}")

# ---- Extract Reference Projection ----
try:
    ref_proj  = s2_collection.first().select('B2').projection()
    proj_info = ref_proj.getInfo()
    print(f"  CRS (reference): {proj_info.get('crs', 'N/A')}")
except Exception as e:
    print(f"  ⚠️  Could not extract projection: {e}")
    print(f"  Will use EPSG:32748 (UTM Zone 48S) as fallback")
    ref_proj = ee.Projection('EPSG:32748')

# ---- Monthly Scene Count Table ----
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

print(f"\n  {'Month':<18s} {'Scenes':>8s}  {'Coverage':>10s}")
print("  " + "-"*40)

monthly_scene_counts = {}

for m in range(1, 13):
    start = ee.Date.fromYMD(YEAR_S2, m, 1)
    end   = start.advance(1, 'month')
    count = s2_collection.filterDate(start, end).size().getInfo()
    monthly_scene_counts[m] = count
    
    # Coverage indicator
    if count >= 5:
        status   = "✅"
        coverage = "Good"
    elif count >= 2:
        status   = "⚠️ "
        coverage = "Fair"
    elif count == 1:
        status   = "🔶"
        coverage = "Minimal"
    else:
        status   = "❌"
        coverage = "No data"
    
    print(f"  {status} Month {m:02d} ({MONTH_NAMES[m-1]:<4s})  "
          f"{count:>8d}  {coverage:>10s}")

print("  " + "-"*40)
print(f"  {'Total':>24s} : {total_scenes:>5d} scenes")
print(f"  {'Months w/ data':>24s} : "
      f"{sum(1 for c in monthly_scene_counts.values() if c > 0):>5d} / 12")


📡 SENTINEL-2 SR HARMONIZED
------------------------------------------------------------
  Collection : COPERNICUS/S2_SR_HARMONIZED
  Year       : 2023
  Cloud max  : 70%
  Region     : Sawit-PDL

  ✅ Collection built
  Total scenes after filter: 68
  CRS (reference): EPSG:32750

  Month                Scenes    Coverage
  ----------------------------------------
  ❌ Month 01 (Jan )         0     No data
  ⚠️  Month 02 (Feb )         2        Fair
  ⚠️  Month 03 (Mar )         4        Fair
  ❌ Month 04 (Apr )         0     No data
  ✅ Month 05 (May )         8        Good
  ✅ Month 06 (Jun )         7        Good
  ✅ Month 07 (Jul )         8        Good
  ✅ Month 08 (Aug )        11        Good
  ✅ Month 09 (Sep )         8        Good
  ✅ Month 10 (Oct )         8        Good
  ✅ Month 11 (Nov )         6        Good
  ✅ Month 12 (Dec )         6        Good
  ----------------------------------------
                     Total :    68 scenes
            Months w/ data :    10 / 12
